### Lumièvre — Audit flash des données (Phase 1 : Cadrage métier)

**Objectif** : pas l'EDA complète (Phase 5), juste un audit rapide du dataset UCI
Online Retail II pour ancrer le cadrage métier dans des chiffres réels.

Étapes :
1. Vue d'ensemble brute du fichier
2. Nettoyage minimal (pour cet audit seulement)
3. Base clients "fidèles" et chiffre d'affaires
4. Distribution des délais entre achats → choix du seuil de churn
5. Taux de churn réel + perte de CA estimée
6. Table de sensibilité pour fixer une cible SMART

In [2]:
import pandas as pd

DATA_PATH = "../data/raw/online_retail_II.xlsx"

xl = pd.ExcelFile(DATA_PATH)
print("Feuilles disponibles :", xl.sheet_names)

Feuilles disponibles : ['Year 2009-2010', 'Year 2010-2011']


#### 1. Vue d'ensemble brute

On charge les deux feuilles et on regarde ce qu'on a réellement, sans rien corriger.

In [3]:
df = pd.concat([xl.parse("Year 2009-2010"), xl.parse("Year 2010-2011")], ignore_index=True)

print("Lignes totales :", len(df))
print("Période couverte :", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())
print("Nb clients uniques (Customer ID renseigné) :", df["Customer ID"].nunique())
print("Nb lignes SANS Customer ID :", df["Customer ID"].isna().sum())
print("Nb pays :", df["Country"].nunique())
print(df["Country"].value_counts().head(5))
print("Quantity min/max :", df["Quantity"].min(), df["Quantity"].max())
print("Price min/max :", df["Price"].min(), df["Price"].max())
print("Nb invoices uniques :", df["Invoice"].nunique())
nb_annulations = df["Invoice"].astype(str).str.startswith("C").sum()
print("Nb lignes d'annulation (Invoice commençant par 'C') :", nb_annulations)

Lignes totales : 1067371
Période couverte : 2009-12-01 07:45:00 -> 2011-12-09 12:50:00
Nb clients uniques (Customer ID renseigné) : 5942
Nb lignes SANS Customer ID : 243007
Nb pays : 43
Country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Name: count, dtype: int64
Quantity min/max : -80995 80995
Price min/max : -53594.36 38970.0
Nb invoices uniques : 53628
Nb lignes d'annulation (Invoice commençant par 'C') : 19494


In [4]:
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 2. Nettoyage minimal (pour cet audit uniquement)

On retire : lignes sans Customer ID, annulations, quantités/prix ≤ 0.
Ce n'est PAS la règle de nettoyage officielle du projet — celle-ci sera décidée
en Phase 5 (EDA complète) et deviendra un modèle `staging` dbt.

In [5]:
df_clean = df.dropna(subset=["Customer ID"]).copy()
df_clean = df_clean[~df_clean["Invoice"].astype(str).str.startswith("C")]
df_clean = df_clean[(df_clean["Quantity"] > 0) & (df_clean["Price"] > 0)]
df_clean["Revenue"] = df_clean["Quantity"] * df_clean["Price"]

print("Lignes après nettoyage minimal :", len(df_clean))

Lignes après nettoyage minimal : 805549


## 3. Clients "fidèles" et chiffre d'affaires

On définit une commande comme (Customer ID, InvoiceDate) distinct — pas la ligne produit.

In [6]:
orders = (
    df_clean.groupby(["Customer ID", "InvoiceDate"])
    .size()
    .reset_index()[["Customer ID", "InvoiceDate"]]
)
nb_orders_par_client = orders.groupby("Customer ID").size()

nb_clients = nb_orders_par_client.shape[0]
nb_fideles = (nb_orders_par_client >= 3).sum()
ca_total = df_clean["Revenue"].sum()
ca_moyen_client = df_clean.groupby("Customer ID")["Revenue"].sum().mean()

print(f"Nb clients (>=1 commande) : {nb_clients}")
print(f"Nb clients fidèles (>=3 commandes) : {nb_fideles}")
print(f"CA total (période ~2 ans) : {ca_total:,.0f} £")
print(f"CA moyen par client sur la période : {ca_moyen_client:,.0f} £")

Nb clients (>=1 commande) : 5878
Nb clients fidèles (>=3 commandes) : 3310
CA total (période ~2 ans) : 17,743,429 £
CA moyen par client sur la période : 3,019 £


## 4. Distribution des délais entre deux achats consécutifs

C'est LE calcul qui sert à choisir le seuil de churn : plutôt que de deviner un nombre
de jours, on regarde comment les clients achètent réellement.

In [7]:
orders_sorted = orders.sort_values(["Customer ID", "InvoiceDate"]).copy()
orders_sorted["prev"] = orders_sorted.groupby("Customer ID")["InvoiceDate"].shift(1)
orders_sorted["delay_days"] = (
    orders_sorted["InvoiceDate"] - orders_sorted["prev"]
).dt.total_seconds() / 86400

delays = orders_sorted["delay_days"].dropna()
print(delays.describe())
for q in [0.5, 0.75, 0.9, 0.95]:
    print(f"  p{int(q*100)} = {delays.quantile(q):.0f} jours")

count    30973.000000
mean        51.883571
std         75.865841
min          0.000694
25%          7.004167
50%         24.904861
75%         61.949306
max        714.152083
Name: delay_days, dtype: float64
  p50 = 25 jours
  p75 = 62 jours
  p90 = 135 jours
  p95 = 207 jours


## Décision retenue

Seuil de churn = **90 jours** sans achat pour un client fidèle (≥3 commandes).
Compromis entre p75 (62j, trop de faux positifs) et p90 (135j, détection trop tardive
par rapport au brief qui évoque "plusieurs mois").
Ce seuil définit la vérité terrain (label historique) pour le futur modèle ML.

In [8]:
SEUIL_CHURN_JOURS = 90

## 5. Taux de churn réel et perte de CA estimée, avec ce seuil

In [9]:
last_date = df_clean["InvoiceDate"].max()
print('Date de référence ("aujourd\'hui" de notre histoire) :', last_date)

last_purchase = orders.groupby("Customer ID")["InvoiceDate"].max()
clients_fideles = nb_orders_par_client[nb_orders_par_client >= 3].index

recency_days = (last_date - last_purchase.loc[clients_fideles]).dt.total_seconds() / 86400
nb_churn = (recency_days > SEUIL_CHURN_JOURS).sum()
taux_churn = 100 * nb_churn / len(clients_fideles)

revenue_par_client = df_clean.groupby("Customer ID")["Revenue"].sum()
ca_moyen_fidele_periode = revenue_par_client.loc[clients_fideles].mean()
ca_moyen_fidele_annuel = ca_moyen_fidele_periode / 2

perte_ca_annualisee = nb_churn * ca_moyen_fidele_annuel

print(f"Nb clients fidèles : {len(clients_fideles)}")
print(f"Nb fidèles en churn (silence > {SEUIL_CHURN_JOURS}j) : {nb_churn}")
print(f"Taux de churn chez les fidèles : {taux_churn:.1f} %")
print(f"CA annuel moyen par client fidèle : {ca_moyen_fidele_annuel:,.0f} £")
print(f"Perte de CA annualisée estimée : {perte_ca_annualisee:,.0f} £")

Date de référence ("aujourd'hui" de notre histoire) : 2011-12-09 12:50:00
Nb clients fidèles : 3310
Nb fidèles en churn (silence > 90j) : 1123
Taux de churn chez les fidèles : 33.9 %
CA annuel moyen par client fidèle : 2,470 £
Perte de CA annualisée estimée : 2,773,664 £


## 6. Table de sensibilité — fixer une cible SMART réaliste

On ne choisit pas la cible de churn au hasard : elle dépend de deux hypothèses non
maîtrisées à ce stade (pas de données de campagne pour l'instant) :
- le **recall** du futur modèle ML (part des vrais churners détectés)
- le **taux de réactivation** de l'action marketing envoyée aux clients détectés

In [10]:
for recall in [0.6, 0.7, 0.8]:
    for reactivation in [0.15, 0.25, 0.35]:
        detectes = nb_churn * recall
        sauves = detectes * reactivation
        nouveau_churn = nb_churn - sauves
        nouveau_taux = 100 * nouveau_churn / len(clients_fideles)
        print(
            f"recall={recall:.0%} x réactivation={reactivation:.0%} "
            f"-> clients sauvés={sauves:.0f} -> nouveau taux de churn={nouveau_taux:.1f}%"
        )

recall=60% x réactivation=15% -> clients sauvés=101 -> nouveau taux de churn=30.9%
recall=60% x réactivation=25% -> clients sauvés=168 -> nouveau taux de churn=28.8%
recall=60% x réactivation=35% -> clients sauvés=236 -> nouveau taux de churn=26.8%
recall=70% x réactivation=15% -> clients sauvés=118 -> nouveau taux de churn=30.4%
recall=70% x réactivation=25% -> clients sauvés=197 -> nouveau taux de churn=28.0%
recall=70% x réactivation=35% -> clients sauvés=275 -> nouveau taux de churn=25.6%
recall=80% x réactivation=15% -> clients sauvés=135 -> nouveau taux de churn=29.9%
recall=80% x réactivation=25% -> clients sauvés=225 -> nouveau taux de churn=27.1%
recall=80% x réactivation=35% -> clients sauvés=314 -> nouveau taux de churn=24.4%
